# Part 2 — Exploratory Data Analysis with Apache Spark
    Uses Spark DataFrames, distributed aggregations, and partition inspection. Run in Colab or a local Spark environment with Java configured.

In [ ]:
# In Google Colab, uncomment and run once if PySpark is not installed:
    # %pip -q install pyspark==3.5.3

In [ ]:
from pyspark.sql import SparkSession, functions as F
    spark = (SparkSession.builder.appName("BankingEDA")
             .config("spark.sql.shuffle.partitions", "4").getOrCreate())
    bank_path = "../data/bank.csv"
    import os
    if not os.path.exists(bank_path):
        bank_path = "data/bank.csv"
    sdf = spark.read.option("header", True).option("inferSchema", True).csv(bank_path)
    print("Rows:", sdf.count(), "Columns:", len(sdf.columns))
    print("Input partitions:", sdf.rdd.getNumPartitions())
    sdf.printSchema()

In [ ]:
# Missing-value profile and target balance
    sdf.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in sdf.columns]).show(truncate=False)
    sdf.groupBy("y").count().orderBy(F.desc("count")).show()

In [ ]:
# Conversion rates by job and month (rates are more useful than raw counts alone)
    job_rates = (sdf.groupBy("job")
                 .agg(F.count("*").alias("customers"),
                      F.sum(F.when(F.col("y") == "yes", 1).otherwise(0)).alias("subscriptions"))
                 .withColumn("conversion_pct", F.round(100*F.col("subscriptions")/F.col("customers"), 2))
                 .orderBy(F.desc("conversion_pct")))
    job_rates.show(truncate=False)
    month_rates = (sdf.groupBy("month")
                   .agg(F.count("*").alias("customers"),
                        F.sum(F.when(F.col("y") == "yes", 1).otherwise(0)).alias("subscriptions"))
                   .withColumn("conversion_pct", F.round(100*F.col("subscriptions")/F.col("customers"), 2))
                   .orderBy(F.desc("conversion_pct")))
    month_rates.show(truncate=False)

In [ ]:
# Preserve the meaning of pdays=-1 rather than treating it as an ordinary elapsed-day value
    sdf = sdf.withColumn("previously_contacted",
                         F.when(F.col("pdays") == -1, "no").otherwise("yes"))
    sdf.groupBy("previously_contacted", "y").count().orderBy("previously_contacted", "y").show()
    sdf.select("age", "balance", "campaign", "duration", "pdays", "previous").describe().show()

In [ ]:
# Data parallelism demonstration: repartitioning by a commonly analyzed category
    print("Before:", sdf.rdd.getNumPartitions())
    repartitioned = sdf.repartition(4, "job")
    print("After:", repartitioned.rdd.getNumPartitions())
    # Repartition introduces a shuffle; more partitions are not always faster.
    spark.stop()

## Interpretation guide
    - Uploaded data contains 4,521 records and 17 columns, with 0 missing cells.
    - Target counts are 4,000 `no` and 521 `yes` (about 11.5% positive).
    - `pdays=-1` means no prior campaign contact; derive an indicator.
    - Spark transformations are lazy; actions such as `count()` and `show()` trigger execution.
    - The dataset is small, so this illustrates distributed APIs but does not prove big-data speedup.